# Gamma Radiation Attenuation in Lead (Pb)

This notebook analyzes the attenuation of gamma rays in lead shielding.

## Experimental Design

**Status**: Template - waiting for measurement data

### Expected Analysis:
1. Measure gamma ray intensity with intermediate lead thickness
2. Measure gamma ray intensity with large lead thickness
3. Extract mass attenuation coefficient μ/ρ
4. Compare with theoretical values (NIST data)
5. Determine half-value layer (HVL)
6. Analyze beam hardening effects

### Theoretical Background

The intensity of gamma rays through matter follows:
$$I = I_0 e^{-\mu x}$$

Where:
- $I_0$ = incident intensity
- $\mu$ = linear attenuation coefficient (cm⁻¹)
- $x$ = thickness (cm)

Or using mass attenuation coefficient:
$$I = I_0 e^{-(\mu/\rho) \rho x}$$

The **half-value layer (HVL)** is the thickness needed to reduce intensity by 50%:
$$\text{HVL} = \frac{\ln(2)}{\mu}$$

In [ ]:
import glob
import os
import re
import matplotlib.pyplot as plt

DATA_GLOB = "PB*.ASC"
OUTPUT_FILE = "PB_overlay.png"


def read_asc_spectrum(path):
    channels = []
    counts = []
    with open(path, "r", encoding="latin1", errors="replace") as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("Chn") or line.startswith("ID:"):
                continue
            parts = re.split(r"[\s,]+", line)
            if len(parts) < 2:
                continue
            try:
                ch = int(parts[0])
                cnt = int(parts[1])
            except ValueError:
                continue
            channels.append(ch)
            counts.append(cnt)
    return channels, counts


def main():
    files = sorted(glob.glob(DATA_GLOB))
    if not files:
        raise FileNotFoundError(f"No files found matching {DATA_GLOB}")

    labels = {
        'PBA.ASC': 'A',
        'PBALL.ASC': 'A+B+C+D+E',
        'PBC.ASC': 'C',
        'PBE.ASC': 'E',
        'PBED.ASC': 'E + D'
    }

    plt.figure(figsize=(12, 7))
    for path in files:
        ch, cnt = read_asc_spectrum(path)
        label = labels.get(os.path.basename(path), os.path.basename(path))
        plt.plot(ch, cnt, label=label, linewidth=1)

    plt.title("Attenuation of Gamma Radiation in Matter")
    plt.xlabel("Channel")
    plt.ylabel("Counts")
    plt.legend(loc="upper right", fontsize="small")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_FILE, dpi=300)
    print(f"Saved overlay plot to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()